In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
from pyspark.sql.types import IntegerType
from pyspark.sql.functions import desc
from pyspark.sql.functions import asc
from pyspark.sql.functions import sum as Fsum

import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
# download from google storage
import requests

url = 'https://storage.googleapis.com/files-bb-bot/music_log.json'
r = requests.get(url)
with open('./data/music_log.json', 'wb') as f:
    f.write(r.content)

In [3]:
spark = SparkSession \
    .builder \
    .appName("fhtw music analytics") \
    .getOrCreate()

26/05/18 16:45:57 WARN Utils: Your hostname, codespaces-4701ab resolves to a loopback address: 127.0.0.1; using 10.0.16.108 instead (on interface eth0)
26/05/18 16:45:57 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/18 16:45:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
df = spark.read.json("./data/music_log.json")

In [5]:
df

DataFrame[artist: string, auth: string, firstName: string, gender: string, itemInSession: bigint, lastName: string, length: double, level: string, location: string, method: string, page: string, registration: bigint, sessionId: bigint, song: string, status: bigint, ts: bigint, userAgent: string, userId: string]

### 1. RDD Query
**Description**: 
**Resilient Distributed Datasets (RDDs)** are the fundamental building blocks of Spark, providing a low-level, fine-grained data processing framework. RDDs are immutable, distributed collections of objects, which means once you create an RDD, you cannot change it. Each RDD is split into multiple partitions, which may be processed in parallel across different nodes of a Spark cluster.

**Use Cases**:
- **Complex Data Processing**: When the data processing task involves complex manipulations that are not supported directly by DataFrame or SQL operations.
- **Custom Partitioning**: RDDs allow you to control how data is distributed across the cluster, which can optimize specific types of operations.
- **Legacy Integration**: In scenarios where you need to integrate with legacy systems or algorithms originally designed for RDDs.

**Advantages**:
- **Flexibility**: Offers complete control over data processing and transformation steps.
- **Fine-grained Operations**: Supports detailed manipulation and processing of data at the element level.

In [6]:
songs_rdd = df.rdd

In [7]:
songs_rdd

MapPartitionsRDD[8] at javaToPython at NativeMethodAccessorImpl.java:0

 **Example Query**:

Assuming you have already loaded your RDD from the music log JSON file
Query to count song plays by each user using RDD:

In [8]:
# Filter to include only song plays, then map and reduce by key
song_plays_by_user_rdd = songs_rdd \
    .filter(lambda x: x.page == 'NextSong') \
    .map(lambda x: (x.userId, 1)) \
    .reduceByKey(lambda x, y: x + y)

# Collect and print the result
collected_song_plays_rdd = song_plays_by_user_rdd.collect()
for user, count in collected_song_plays_rdd:
    print(f"User ID {user} has played {count} songs.")

User ID 4 has played 2048 songs.
User ID 78 has played 254 songs.
User ID 142 has played 1875 songs.
User ID 13 has played 1280 songs.
User ID 45 has played 1484 songs.
User ID 131 has played 1564 songs.
User ID 126 has played 2577 songs.
User ID 15 has played 1914 songs.
User ID 33 has played 1257 songs.
User ID 51 has played 2111 songs.
User ID 140 has played 5664 songs.
User ID 6 has played 3159 songs.
User ID 124 has played 4079 songs.
User ID 40 has played 1078 songs.
User ID 38 has played 1322 songs.
User ID 58 has played 1694 songs.
User ID 29 has played 3028 songs.
User ID 70 has played 1490 songs.
User ID 61 has played 1622 songs.
User ID 77 has played 1047 songs.
User ID 136 has played 2124 songs.
User ID 80 has played 367 songs.
User ID 110 has played 178 songs.
User ID 121 has played 726 songs.
User ID 117 has played 343 songs.
User ID 18 has played 429 songs.
User ID 96 has played 1802 songs.
User ID 55 has played 381 songs.
User ID 3 has played 214 songs.
User ID 73 has p

### 2. DataFrame Query
**Description**:
**DataFrames** are a more modern abstraction in Spark, designed to simplify working with structured and semi-structured data. Like RDDs, DataFrames are immutable and distributed, but they are organized into named columns, much like a table in a relational database. This allows Spark to apply advanced optimizations based on the schema.

**Use Cases**:
- **Interactive Data Analysis**: Ideal for ad-hoc queries and exploratory data analysis because of their simplicity and built-in functions.
- **Machine Learning Pipelines**: DataFrames integrate seamlessly with MLlib, Spark's machine learning library, to preprocess data and train models.
- **Stream Processing**: Easily integrates with Structured Streaming to process real-time data streams.

**Advantages**:
- **Performance**: Catalyst optimizer improves performance by optimizing execution plans.
- **Ease of Use**: High-level API for complex transformations and aggregations.
- **Integration**: Supports a wide range of data sources and formats for input and output.

**Example Query**:


In [ ]:
from pyspark.sql.functions import col

# Assuming 'df' is the DataFrame loaded from the music log JSON file
# Count song plays by each user using DataFrame API
song_plays_by_user_df = df \
    .filter(col('page') == 'NextSong') \
    .groupBy('userId') \
    .count() \
    .withColumnRenamed('count', 'song_plays') #not necessary, but nice

song_plays_by_user_df.show()

+------+----------+
|userId|song_plays|
+------+----------+
|   125|         8|
|    51|      2111|
|   124|      4079|
|     7|       150|
|    54|      2841|
|    15|      1914|
|   155|       820|
|   132|      1928|
|   154|        84|
|   101|      1797|
|    11|       647|
|   138|      2070|
|    29|      3028|
|    69|      1125|
|    42|      3573|
|   112|       215|
|    87|       767|
|    73|       377|
|    64|        46|
|     3|       214|
+------+----------+
only showing top 20 rows



### 3. Spark SQL Query
**Description**:
**Spark SQL** is a module in Spark that allows for SQL and HiveQL querying syntax. It provides seamless integration between relational and procedural data manipulations, working across both DataFrames and Datasets. This module not only supports traditional SQL queries but also enables developers to intermix SQL queries with the programmatic data manipulations of DataFrames.

**Use Cases**:
- **SQL and Hive Users**: Offers a familiar interface for SQL and Hive users to run queries on big data.
- **Data Warehousing**: Can be used as part of a larger data warehousing setup that integrates with existing Hive setups.
- **Unified Data Access**: Query data directly from multiple sources, including JSON, Hive, Avro, Parquet, and ORC.

**Advantages**:
- **Optimization**: Benefits from Catalyst optimizer and Tungsten execution engine for efficient query execution.
- **Unified API**: Allows mixing SQL with functional programming, making complex workflows easier to develop and debug.

**Example Query**:

In [ ]:
# Register the DataFrame as a temporary view
df.createOrReplaceTempView("music_logs") #name of the view

# Use Spark SQL to perform the same count
song_plays_by_user_sql = spark.sql("""
SELECT userId, COUNT(*) AS song_plays
FROM music_logs
WHERE page = 'NextSong'


GROUP BY userId
ORDER BY song_plays DESC
""")

song_plays_by_user_sql.show()

+------+----------+
|userId|song_plays|
+------+----------+
|    39|      8002|
|    92|      5945|
|   140|      5664|
|300011|      4619|
|   124|      4079|
|300021|      3816|
|300017|      3632|
|    85|      3616|
|    42|      3573|
|     6|      3159|
|    29|      3028|
|200023|      2955|
|    54|      2841|
|   100|      2682|
|     9|      2676|
|    91|      2580|
|   126|      2577|
|300015|      2524|
|    98|      2401|
|    74|      2400|
+------+----------+
only showing top 20 rows



### Assignment:
Analyze user engagement by calculating metrics **per user**: `total songs played`, `total thumbs up given`, and `songs added to playlists`.
- using SparkSQL
- using DataFrames
- using RDD

**Using SparkSQL**

In [11]:
# Register the DataFrame as a temporary view
df.createOrReplaceTempView("music_metrics") #name of the view

# Use Spark SQL to perform the same count
song_metrics_by_user_sql = spark.sql("""
SELECT
    userId,
    SUM(CASE WHEN page = 'NextSong' THEN 1 ELSE 0 END) AS total_songs_played,
    SUM(CASE WHEN page = 'Thumbs Up' THEN 1 ELSE 0 END) AS total_thumbs_up,
    SUM(CASE WHEN page = 'Add to Playlist' THEN 1 ELSE 0 END) AS songs_added_to_playlists
FROM music_metrics
WHERE userId IS NOT NULL AND userId != ''
GROUP BY userId
ORDER BY userId
""")

song_metrics_by_user_sql.show()

+------+------------------+---------------+------------------------+
|userId|total_songs_played|total_thumbs_up|songs_added_to_playlists|
+------+------------------+---------------+------------------------+
|    10|               673|             37|                       9|
|   100|              2682|            148|                      61|
|100001|               133|              8|                       3|
|100002|               195|              5|                       5|
|100003|                51|              3|                       2|
|100004|               942|             35|                      23|
|100005|               154|              7|                       3|
|100006|                26|              2|                       1|
|100007|               423|             19|                       9|
|100008|               772|             37|                      30|
|100009|               518|             23|                      12|
|100010|               275|       

In [22]:
# in class
df.createOrReplaceTempView("events") #df from json

songs_played = "SELECT userId, COUNT(*) AS songs_played FROM events where page = 'NextSong' GROUP BY userId"
thumbs_up = "SELECT userId, COUNT(*) AS thumbs_up FROM events where LOWER(page) = 'thumbs up' GROUP BY userId"

songs_played_df1 = spark.sql(songs_played)
thumbs_up_df1 = spark.sql(thumbs_up)

songs_played_df1.createOrReplaceTempView("songs_view")
thumbs_up_df1.createOrReplaceTempView("thumbs_up_view")

In [23]:
joined_df = spark.sql("""
                      
    SELECT s.userId, s.songs_played, t.thumbs_up
    FROM songs_view s
                      
    FULL OUTER JOIN thumbs_up_view t
    ON s.userId = t.userId
    """)

In [24]:
joined_df.show()

+------+------------+---------+
|userId|songs_played|thumbs_up|
+------+------------+---------+
|    10|         673|       37|
|   100|        2682|      148|
|100001|         133|        8|
|100002|         195|        5|
|100003|          51|        3|
|100004|         942|       35|
|100005|         154|        7|
|100006|          26|        2|
|100007|         423|       19|
|100008|         772|       37|
|100009|         518|       23|
|100010|         275|       17|
|100011|          11|     NULL|
|100012|         476|       18|
|100013|        1131|       39|
|100014|         257|       17|
|100015|         800|       35|
|100016|         530|       25|
|100017|          52|        2|
|100018|        1002|       46|
+------+------------+---------+
only showing top 20 rows



In [28]:
joined_df_1 = spark.sql("""
    SELECT 
        s.userId, 
        s.songs_played, 
        t.thumbs_up
    FROM songs_view s
    FULL OUTER JOIN thumbs_up_view t
        ON s.userId = t.userId
    WHERE s.userId = 10
""")

In [29]:
joined_df_1.show()

+------+------------+---------+
|userId|songs_played|thumbs_up|
+------+------------+---------+
|    10|         673|       37|
+------+------------+---------+



In [ ]:
# songs_played_df1.show()
# songs_played_df1.explain(True)

**Using DataFrames**

In [14]:
from pyspark.sql.functions import col

from pyspark.sql.functions import col

songs_played_metric_df = (
    df.filter(col("page") == "NextSong")
      .groupBy("userId")
      .count()
      .withColumnRenamed("count", "songs_played")
)

# songs_played_metric_df.show()

thumbs_up_metric_df = (
    df.filter(col("page") == "Thumbs Up")
      .groupBy("userId")
      .count()
      .withColumnRenamed("count", "thumbs_up")
)

# thumbs_up_metric_df.show()

playlist_adds_metric_df = (
    df.filter(col("page") == "Add to Playlist")
      .groupBy("userId")
      .count()
      .withColumnRenamed("count", "playlist_adds")
)

# playlist_adds_metric_df.show()

all_metrics_df = (
    songs_played_metric_df
    .join(thumbs_up_metric_df, on="userId", how="outer")
    .join(playlist_adds_metric_df, on="userId", how="outer")
    .fillna(0)
)

all_metrics_df.show()


+------+------------+---------+-------------+
|userId|songs_played|thumbs_up|playlist_adds|
+------+------------+---------+-------------+
|    10|         673|       37|            9|
|   100|        2682|      148|           61|
|100001|         133|        8|            3|
|100002|         195|        5|            5|
|100003|          51|        3|            2|
|100004|         942|       35|           23|
|100005|         154|        7|            3|
|100006|          26|        2|            1|
|100007|         423|       19|            9|
|100008|         772|       37|           30|
|100009|         518|       23|           12|
|100010|         275|       17|            7|
|100011|          11|        0|            2|
|100012|         476|       18|           12|
|100013|        1131|       39|           31|
|100014|         257|       17|            7|
|100015|         800|       35|           22|
|100016|         530|       25|            6|
|100017|          52|        2|   

**Using RDD**

In [16]:
# Songs played (NextSong)
songs_played_rdd = (
    df.rdd
      .filter(lambda x: x.page == "NextSong")
      .map(lambda x: (x.userId, 1))
      .reduceByKey(lambda a, b: a + b)
)

for user, count in songs_played_rdd.collect():
    print(f"User {user} played {count} songs.")

# Thumbs Up
thumbs_up_rdd = (
    df.rdd
      .filter(lambda x: x.page == "Thumbs Up")
      .map(lambda x: (x.userId, 1))
      .reduceByKey(lambda a, b: a + b)
)

for user, count in thumbs_up_rdd.collect():
    print(f"User {user} gave {count} thumbs up.")

# Add to Playlist
playlist_adds_rdd = (
    df.rdd
      .filter(lambda x: x.page == "Add to Playlist")
      .map(lambda x: (x.userId, 1))
      .reduceByKey(lambda a, b: a + b)
)

for user, count in playlist_adds_rdd.collect():
    print(f"User {user} added {count} songs to playlists.")

# Join RDDs on userId
engagement_rdd = (
    songs_played_rdd
    .fullOuterJoin(thumbs_up_rdd)
    .fullOuterJoin(playlist_adds_rdd)
    .map(lambda x: (
        x[0],
        x[1][0][0] if x[1][0] else 0,   # songs played
        x[1][0][1] if x[1][0] else 0,   # thumbs up
        x[1][1] if x[1][1] else 0       # playlist adds
    ))
)

for user, songs, thumbs, adds in engagement_rdd.collect():
    print(f"User {user}: songs={songs}, thumbs_up={thumbs}, playlist_adds={adds}")

User 4 played 2048 songs.
User 78 played 254 songs.
User 142 played 1875 songs.
User 13 played 1280 songs.
User 45 played 1484 songs.
User 131 played 1564 songs.
User 126 played 2577 songs.
User 15 played 1914 songs.
User 33 played 1257 songs.
User 51 played 2111 songs.
User 140 played 5664 songs.
User 6 played 3159 songs.
User 124 played 4079 songs.
User 40 played 1078 songs.
User 38 played 1322 songs.
User 58 played 1694 songs.
User 29 played 3028 songs.
User 70 played 1490 songs.
User 61 played 1622 songs.
User 77 played 1047 songs.
User 136 played 2124 songs.
User 80 played 367 songs.
User 110 played 178 songs.
User 121 played 726 songs.
User 117 played 343 songs.
User 18 played 429 songs.
User 96 played 1802 songs.
User 55 played 381 songs.
User 3 played 214 songs.
User 73 played 377 songs.
User 17 played 927 songs.
User 7 played 150 songs.
User 32 played 80 songs.
User 114 played 1292 songs.
User 16 played 675 songs.
User 24 played 482 songs.
User 103 played 1073 songs.
User 102 

User 78 gave 11 thumbs up.
User 142 gave 111 thumbs up.
User 131 gave 72 thumbs up.
User 126 gave 135 thumbs up.
User 33 gave 79 thumbs up.
User 51 gave 100 thumbs up.
User 140 gave 277 thumbs up.
User 40 gave 66 thumbs up.
User 124 gave 171 thumbs up.
User 58 gave 94 thumbs up.
User 70 gave 90 thumbs up.
User 29 gave 154 thumbs up.
User 61 gave 78 thumbs up.
User 13 gave 57 thumbs up.
User 77 gave 46 thumbs up.
User 80 gave 11 thumbs up.
User 110 gave 14 thumbs up.
User 121 gave 28 thumbs up.
User 117 gave 11 thumbs up.
User 18 gave 20 thumbs up.
User 96 gave 92 thumbs up.
User 3 gave 14 thumbs up.
User 73 gave 14 thumbs up.
User 15 gave 81 thumbs up.
User 32 gave 7 thumbs up.
User 114 gave 74 thumbs up.
User 16 gave 42 thumbs up.
User 103 gave 52 thumbs up.
User 102 gave 6 thumbs up.
User 75 gave 48 thumbs up.
User 6 gave 165 thumbs up.
User 112 gave 9 thumbs up.
User 83 gave 69 thumbs up.
User 44 gave 25 thumbs up.
User 91 gave 124 thumbs up.
User 148 gave 28 thumbs up.
User 107 gav

User 78 added 9 songs to playlists.
User 13 added 37 songs to playlists.
User 142 added 50 songs to playlists.
User 131 added 51 songs to playlists.
User 126 added 72 songs to playlists.
User 15 added 59 songs to playlists.
User 33 added 33 songs to playlists.
User 51 added 52 songs to playlists.
User 6 added 83 songs to playlists.
User 140 added 148 songs to playlists.
User 40 added 39 songs to playlists.
User 124 added 118 songs to playlists.
User 58 added 50 songs to playlists.
User 70 added 41 songs to playlists.
User 29 added 89 songs to playlists.
User 61 added 50 songs to playlists.
User 80 added 13 songs to playlists.
User 18 added 14 songs to playlists.
User 96 added 52 songs to playlists.
User 121 added 20 songs to playlists.
User 55 added 13 songs to playlists.
User 73 added 11 songs to playlists.
User 136 added 66 songs to playlists.
User 32 added 7 songs to playlists.
User 16 added 19 songs to playlists.
User 75 added 27 songs to playlists.
User 44 added 10 songs to playli

User 4: songs=2048, thumbs_up=95, playlist_adds=59
User 78: songs=254, thumbs_up=11, playlist_adds=9
User 142: songs=1875, thumbs_up=111, playlist_adds=50
User 13: songs=1280, thumbs_up=57, playlist_adds=37
User 45: songs=1484, thumbs_up=67, playlist_adds=43
User 131: songs=1564, thumbs_up=72, playlist_adds=51
User 126: songs=2577, thumbs_up=135, playlist_adds=72
User 15: songs=1914, thumbs_up=81, playlist_adds=59
User 33: songs=1257, thumbs_up=79, playlist_adds=33
User 51: songs=2111, thumbs_up=100, playlist_adds=52
User 140: songs=5664, thumbs_up=277, playlist_adds=148
User 6: songs=3159, thumbs_up=165, playlist_adds=83
User 124: songs=4079, thumbs_up=171, playlist_adds=118
User 40: songs=1078, thumbs_up=66, playlist_adds=39
User 38: songs=1322, thumbs_up=65, playlist_adds=30
User 58: songs=1694, thumbs_up=94, playlist_adds=50
User 29: songs=3028, thumbs_up=154, playlist_adds=89
User 70: songs=1490, thumbs_up=90, playlist_adds=41
User 61: songs=1622, thumbs_up=78, playlist_adds=50
Us